In [1]:
import numpy as np
import polars as pl
from pathlib import Path
import hashlib
import json
from pathlib import Path
from sklearn.preprocessing import LabelEncoder

In [2]:
schema = {
    "section id": pl.Utf8,
    "recording id": pl.Utf8,

    "timestamp [ns]": pl.Int64,
    "gaze x [px]": pl.Float64,
    "gaze y [px]": pl.Float64,

    "fixation id": pl.Utf8,
    "blink id": pl.Utf8,
    "saccade id": pl.Utf8,

    "azimuth [deg]": pl.Float64,
    "elevation [deg]": pl.Float64,

    "start timestamp [ns]": pl.Int64,
    "end timestamp [ns]": pl.Int64,

    "duration [ms]": pl.Int64,
    "amplitude [px]": pl.Float64,
    "amplitude [deg]": pl.Float64,
    "mean velocity [px/s]": pl.Float64,
    "peak velocity [px/s]": pl.Float64,

    "timestamp [ms]": pl.Int64,

    "label": pl.Utf8,
    "norm_time": pl.Float64,
    "label_right": pl.Utf8,

    "duration [ms] fix": pl.Int64,
    "fixation x [px]": pl.Float64,
    "fixation y [px]": pl.Float64,

    "duration [ms] sacc": pl.Int64,
    "amplitude [deg] sacc": pl.Float64,
    "mean velocity [px/s] sacc": pl.Float64,
    "peak velocity [px/s] sacc": pl.Float64,

    "duration [ms] blink": pl.Int64,

    "gyro x [deg/s]": pl.Float64,
    "gyro y [deg/s]": pl.Float64,
    "gyro z [deg/s]": pl.Float64,

    "acceleration x [g]": pl.Float64,
    "acceleration y [g]": pl.Float64,
    "acceleration z [g]": pl.Float64,

    "roll [deg]": pl.Float64,
    "pitch [deg]": pl.Float64,
    "yaw [deg]": pl.Float64,

    "quaternion w": pl.Float64,
    "quaternion x": pl.Float64,
    "quaternion y": pl.Float64,
    "quaternion z": pl.Float64,

    "pupil diameter left [mm]": pl.Float64,
    "pupil diameter right [mm]": pl.Float64,

    "eyeball center left x [mm]": pl.Float64,
    "eyeball center left y [mm]": pl.Float64,
    "eyeball center left z [mm]": pl.Float64,

    "eyeball center right x [mm]": pl.Float64,
    "eyeball center right y [mm]": pl.Float64,
    "eyeball center right z [mm]": pl.Float64,

    "optical axis left x": pl.Float64,
    "optical axis left y": pl.Float64,
    "optical axis left z": pl.Float64,

    "optical axis right x": pl.Float64,
    "optical axis right y": pl.Float64,
    "optical axis right z": pl.Float64,

    "eyelid angle top left [rad]": pl.Float64,
    "eyelid angle bottom left [rad]": pl.Float64,
    "eyelid aperture left [mm]": pl.Float64,

    "eyelid angle top right [rad]": pl.Float64,
    "eyelid angle bottom right [rad]": pl.Float64,
    "eyelid aperture right [mm]": pl.Float64,
}

In [3]:
feature_cols = [
    "gaze x [px]",
    "gaze y [px]",
    "azimuth [deg]",
    "elevation [deg]",
    #"pupil diameter left [mm]",
    #"pupil diameter right [mm]",
    
    "optical axis left x",
    "optical axis left y",
    "optical axis left z",
    "optical axis right x",
    "optical axis right y",
    "optical axis right z",

    "gyro x [deg/s]",
    "gyro y [deg/s]",
    "gyro z [deg/s]",
    "acceleration x [g]",
    "acceleration y [g]",
    "acceleration z [g]",
    
    "roll [deg]",
    "pitch [deg]",
    "yaw [deg]",

    #"fixation id",
    #"blink id",
    #"saccade id",
    "duration [ms]",
    "fixation x [px]",
    "fixation y [px]",
    "duration [ms] sacc",
    "amplitude [deg]",
    "mean velocity [px/s]",
    "peak velocity [px/s]",
    "duration [ms] blink",
    # "quaternion w",
    # "quaternion x",
    # "quaternion y",
    # "quaternion z",
    # "eyeball center left x [mm]",
    # "eyeball center left y [mm]",
    # "eyeball center left z [mm]",
    # "eyeball center right x [mm]",
    # "eyeball center right y [mm]",
    # "eyeball center right z [mm]",
    # "eyelid angle top left [rad]",
    # "eyelid angle bottom left [rad]",
    # "eyelid aperture left [mm]",
    # "eyelid angle top right [rad]",
    # "eyelid angle bottom right [rad]",
    # "eyelid aperture right [mm]",
]

In [4]:
def load_or_create_parquet(
    parquet_path,
    csv_path,
    schema=None
):
    parquet_path = Path(parquet_path)

    if parquet_path.exists():
        print("Loading parquet...")
        return pl.scan_parquet(parquet_path)

    print("Creating parquet from CSV...")

    df = pl.read_csv(
        csv_path,
        schema_overrides=schema
    )

    df.write_parquet(parquet_path)

    df = (pl.scan_parquet(parquet_path).selectc(feature_cols))

    return df

In [5]:
data_path = '../../data/'
training_data_path = data_path + "Final Training Data/"
merged_df = load_or_create_parquet(training_data_path + "merged_output.parquet", training_data_path + "merged_output.csv")
to_label_df = pl.read_csv(data_path + "Final Data For Labeling/" + "merged_output.csv")

labels = merged_df.select("label").unique().collect().to_series().to_numpy()

Loading parquet...


In [6]:
le = LabelEncoder()
le.fit(labels)

label_map = dict(zip(le.classes_, range(len(le.classes_))))
    
merged_df = merged_df.with_columns(
    pl.col("label").replace_strict(label_map).alias("label")
)

In [7]:
def get_cache_dir(training_data_path, settings, feature_cols):
    config = {
        "window_size": settings["window size"],
        "overlap": settings["overlap"],
        "target_length": settings["target length"],
        "min_samples": settings["min samples"],
        "features": feature_cols
    }

    config_str = json.dumps(config, sort_keys=True).encode()
    config_hash = hashlib.md5(config_str).hexdigest()

    cache_dir = Path(training_data_path) / "Windowed Data" / config_hash
    cache_dir.mkdir(parents=True, exist_ok=True)

    return cache_dir

In [8]:
def build_cache(subjects, merged_df, feature_cols, settings, cache_dir):
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)

    for s in subjects:
        path = cache_dir / f"{s}.npz"

        if path.exists():
            print(f"[CACHE] {s} exists")
            continue

        print(f"[BUILD] {s}")

        df_s = merged_df.filter(pl.col("recording id") == s).collect(engine="streaming")

        X, y = window_subject_raw(
            df_s,
            feature_cols,
            "label",
            settings
        )

        np.savez_compressed(path, X=X, y=y)

In [9]:
def window_subject_raw(
    df,
    feature_cols,
    label_col,
    settings
):
    window_size = int(settings["window size"] * 1e9)
    target_length = settings["target length"]
    stride = int(window_size * (1 - settings["overlap"]))

    df = df.sort("timestamp [ns]")

    times = df["timestamp [ns]"].to_numpy()
    features = df.select(feature_cols).to_numpy()
    labels = df[label_col].to_numpy()

    max_time = times[-1]
    new_t = np.linspace(0, 1, target_length, dtype=np.float32)

    X, y = [], []

    start = times[0]

    while start + window_size <= max_time:
        end = start + window_size

        start_idx = np.searchsorted(times, start, side="left")
        end_idx = np.searchsorted(times, end, side="left")

        if end_idx - start_idx < settings["min samples"]:
            start += stride
            continue

        window = features[start_idx:end_idx]

        if np.isnan(window).any():
            start += stride
            continue

        label = labels[start_idx]

        window_times = times[start_idx:end_idx]

        denom = window_times[-1] - window_times[0]
        if denom == 0:
            start += stride
            continue

        old_t = (window_times - window_times[0]) / denom

        resampled = np.empty((target_length, window.shape[1]), dtype=np.float32)

        for i in range(window.shape[1]):
            resampled[:, i] = np.interp(new_t, old_t, window[:, i])

        X.append(resampled)
        y.append(label)

        start += stride

    return np.asarray(X, dtype=np.float32), np.asarray(y)

In [10]:
settings = {
    "window size": 0.22,
    "overlap": 0.0,
    "target length": 45, # Sample rate is 200 Hz and window size is 0.22 so 45 is around 22% of 200
    "min samples": 5
}


subjects = (
    merged_df.select("recording id")
    .unique()
    .sort("recording id")
    .collect(engine="streaming")
    .to_series()
    .to_list()
)


cache_dir = get_cache_dir(training_data_path, settings, feature_cols)
print(f"Using cache dir: {cache_dir}")

build_cache(
    subjects=subjects,
    merged_df=merged_df,
    feature_cols=feature_cols,
    settings=settings,
    cache_dir=cache_dir
)

Using cache dir: ../../data/Final Training Data/Windowed Data/fa58a95392aed78461eb668748d46ab8
[BUILD] 0fc57dc8-de68-4b41-bf69-59a5a5f2d27e
[BUILD] 10b8d8be-39de-43e7-9260-ba4cc724d4ae
[BUILD] 1279952d-14d4-4e77-9010-a000dd546bcd
[BUILD] 1477b633-8ef3-4123-9ecc-f2ee90d13d8c
[BUILD] 383b0fe8-654a-4f4f-84cd-375470069789
[BUILD] 3b1c507b-c290-4250-95c9-e21d6c52e3f2
[BUILD] 5bcd3401-e378-482f-b9c6-37edbad97d1a
[BUILD] 63d66e1c-434f-4f23-8e74-abd8dedc0e43
[BUILD] 63efffc7-e347-47fb-9615-f7da183d8792
[BUILD] 6f32fcb3-9e76-448d-9de8-799e376fdea6
[BUILD] 71748750-99ec-4f47-a314-ebb573f9769d
[BUILD] 853a8f80-6e9a-4e3b-9312-522e2ec6f822
[BUILD] 8850342c-636c-4d0d-b6a8-a9e612e6be45
[BUILD] 911806b6-27bd-4b56-bd2f-45d979842721
[BUILD] 97b2cb35-e6ef-4ee4-b532-5d26cb8aabaa
[BUILD] a99f1fcb-8ae2-4a27-b995-2eb4d46c7c81
[BUILD] b5960073-9188-445e-b7db-cfe89eef979d
[BUILD] b636a895-09a2-4fe2-9c37-973ed9687a60
[BUILD] c532ed90-f0e5-4edf-84e5-57a05bb00823
[BUILD] e2e75728-4988-49c4-b2cc-9ec7a8bd96bc
[BUIL